# Canadian municipal geospatial data: Vancouver parks and local areas
Contains information licensed under the Open Government Licence - Vancouver.
Sources: https://opendata.vancouver.ca/explore/dataset/parks/
and https://opendata.vancouver.ca/explore/dataset/local-area-boundary/
Licence: https://opendata.vancouver.ca/pages/licence/
These are public park locations and planning boundaries, not police data.
WGS84/EPSG:4326; WKT and GeoJSON use longitude, latitude.


In [ ]:
%pip install shapely==2.2.0


In [ ]:
import hashlib
import json
import math
import re
import time
from datetime import datetime, timezone

import requests
from shapely.geometry import Point, shape


CATALOG = "https://opendata.vancouver.ca/api/explore/v2.1/catalog/datasets"
ATTRIBUTION = "Contains information licensed under the Open Government Licence - Vancouver."
LICENSE_URL = "https://opendata.vancouver.ca/pages/licence/"


def fetch_records(dataset, session):
    rows = []
    expected = None
    while expected is None or len(rows) < expected:
        for attempt in range(4):
            response = session.get(
                f"{CATALOG}/{dataset}/records",
                params={"limit": 100, "offset": len(rows)},
                timeout=60,
            )
            if response.status_code not in (429, 500, 502, 503, 504) or attempt == 3:
                break
            time.sleep(2**attempt)
        response.raise_for_status()
        page = response.json()
        if expected is None:
            expected = page["total_count"]
            if not 0 < expected <= 10000:
                raise ValueError("Source size outside the bounded demo limit.")
        if not page["results"]:
            raise ValueError("Source pagination ended before total_count.")
        rows.extend(page["results"])
    if len(rows) != expected:
        raise ValueError("Source changed during pagination; rerun for a complete snapshot.")
    return rows


def area_key(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def distance_km(latitude, longitude, reference_latitude=49.2609, reference_longitude=-123.1139):
    dlat = math.radians(latitude - reference_latitude)
    dlon = math.radians(longitude - reference_longitude)
    a = math.sin(dlat / 2) ** 2 + math.cos(math.radians(latitude)) * math.cos(math.radians(reference_latitude)) * math.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * math.asin(math.sqrt(min(1.0, max(0.0, a))))


def transform(parks, areas):
    polygons = []
    for record in areas:
        geometry = record["geom"]
        polygon = shape(geometry["geometry"] if geometry["type"] == "Feature" else geometry)
        if polygon.is_empty or not polygon.is_valid:
            raise ValueError("Invalid source planning-area polygon.")
        polygons.append((area_key(record["name"]), record["name"], polygon))
    if len({a[0] for a in polygons}) != len(polygons):
        raise ValueError("Duplicate local-area keys.")
    rows, rejected = [], []
    for record in parks:
        location = record.get("googlemapdest") or {}
        latitude, longitude = location.get("lat"), location.get("lon")
        if latitude is None or longitude is None:
            rejected.append({"park_id": record.get("parkid"), "reason": "missing_coordinates"})
            continue
        latitude, longitude = float(latitude), float(longitude)
        if not math.isfinite(latitude) or not math.isfinite(longitude) or not (-90 <= latitude <= 90 and -180 <= longitude <= 180):
            rejected.append({"park_id": record.get("parkid"), "reason": "invalid_coordinates"})
            continue
        point = Point(longitude, latitude)
        matches = sorted(key for key, _, polygon in polygons if polygon.covers(point))
        # Boundary ties are deterministic; unmatched points remain explicitly unassigned.
        key = matches[0] if matches else "unassigned"
        hectares = float(record["hectare"]) if record.get("hectare") is not None else None
        if hectares is not None and (not math.isfinite(hectares) or hectares < 0):
            raise ValueError("Invalid source park area.")
        rows.append((
            int(record["parkid"]), record["name"], key, latitude, longitude,
            hectares, record.get("facilities") == "Y", record.get("washrooms") == "Y",
            distance_km(latitude, longitude), point.wkt,
            "https://opendata.vancouver.ca/explore/dataset/parks/",
        ))
    if not rows or len({r[0] for r in rows}) != len(rows):
        raise ValueError("No usable parks or duplicate source park IDs.")
    dimensions = [(key, name, polygon.wkt) for key, name, polygon in polygons]
    if any(row[2] == "unassigned" for row in rows):
        dimensions.append(("unassigned", "Unassigned by spatial join", None))
    return rows, dimensions, rejected


def main():
    from pyspark.sql import functions as F
    import notebookutils

    spark.conf.set("spark.sql.session.timeZone", "UTC")
    with requests.Session() as session:
        parks = fetch_records("parks", session)
        areas = fetch_records("local-area-boundary", session)
        metadata = {}
        for dataset in ("parks", "local-area-boundary"):
            response = session.get(f"{CATALOG}/{dataset}", timeout=60)
            response.raise_for_status()
            metadata[dataset] = response.json()["metas"]["default"]
    rows, dimensions, rejected = transform(parks, areas)
    notebookutils.fs.mkdirs("Files/bronze/canada")
    notebookutils.fs.mkdirs("Files/geo")
    for name, records in (("parks", parks), ("local_areas", areas)):
        text = json.dumps(records, ensure_ascii=False)
        notebookutils.fs.put(f"Files/bronze/canada/{name}.json", text, True)
        bronze = spark.createDataFrame([(json.dumps(record, ensure_ascii=False),) for record in records], "raw_json string")
        (bronze.withColumn("_load_ts", F.current_timestamp())
            .write.format("delta").mode("overwrite").saveAsTable(f"dbo.bronze_{name}"))
    parks_schema = "park_id long, park_name string, area_id string, latitude double, longitude double, hectares double, facilities boolean, washrooms boolean, distance_km double, geometry_wkt string, source_url string"
    silver = spark.createDataFrame(rows, parks_schema).withColumn("_load_ts", F.current_timestamp())
    silver.write.format("delta").mode("overwrite").saveAsTable("dbo.silver_parks")
    # Full bounded-source snapshots are replay-safe; no append duplicates are introduced.
    silver.write.format("delta").mode("overwrite").saveAsTable("dbo.gold_parks")
    dimension = spark.createDataFrame(dimensions, "area_id string, area_name string, geometry_wkt string")
    dimension.write.format("delta").mode("overwrite").saveAsTable("dbo.dim_local_areas")
    features = []
    for record in areas:
        geometry = record["geom"]
        features.append({
            "type": "Feature",
            "properties": {"area_id": area_key(record["name"]), "area_name": record["name"]},
            "geometry": geometry["geometry"] if geometry["type"] == "Feature" else geometry,
        })
    geojson = {"type": "FeatureCollection", "attribution": ATTRIBUTION, "features": features}
    notebookutils.fs.put("Files/geo/vancouver_local_areas.geojson", json.dumps(geojson), True)
    quality = {
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "attribution": ATTRIBUTION, "license_url": LICENSE_URL,
        "source_parks": len(parks), "source_local_areas": len(areas),
        "gold_parks": len(rows), "dimension_rows": len(dimensions),
        "spatially_assigned_parks": sum(row[2] != "unassigned" for row in rows),
        "unassigned_parks": sum(row[2] == "unassigned" for row in rows),
        "total_hectares": round(sum(row[5] or 0 for row in rows), 4),
        "rejected": rejected,
        "source_modified": {name: value.get("modified") for name, value in metadata.items()},
        "source_sha256": hashlib.sha256(json.dumps({"parks": parks, "areas": areas}, sort_keys=True).encode()).hexdigest(),
    }
    assert spark.table("dbo.gold_parks").count() == len(rows)
    assert spark.table("dbo.dim_local_areas").select("area_id").distinct().count() == len(dimensions)
    assert silver.join(dimension, "area_id", "left_anti").count() == 0
    notebookutils.fs.put("Files/geo/validation.json", json.dumps(quality, indent=2), True)
    print(json.dumps(quality, indent=2))
    display(spark.table("dbo.gold_parks").orderBy("park_id"))


if __name__ == "__main__":
    main()
